In [1]:
# llmcompressor 설치
!pip install -q llmcompressor peft datasets==4.4.1 accelerate==1.10.1

# 대회 규정 버전으로 강제 고정
!pip install -q transformers==4.57.3

# 설치 버전 최종 확인
import transformers
import torch
print(f"Transformers 버전: {transformers.__version__}") # 4.57.3 확인
print(f"Torch 버전: {torch.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.1/196.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 125.7 MB/s eta 0:00:00
Transformers 버전: 4.57.3
Torch 버전: 2.10

In [2]:
#버전 충돌 방지(초반에는 필요 없었는데 코렙 자동 업데이트 때문에 필요함)
!pip uninstall torchvision -y -q
#완료 후 재실행 control+m+.

In [3]:
#  드라이브에서 원본 파일 불러오기
from google.colab import drive
drive.mount('/content/drive')

#  파일 복사
!cp "/content/drive/MyDrive/open.zip" /content/open.zip

#  압축 해제
!unzip -q /content/open.zip -d /content/

#  경로 확인
!ls -F /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
baseline_submit.zip  base_model/  drive/  open.zip  sample_data/


In [4]:
#다시 확인
import os

MODEL_ID = "/content/base_model"

if os.path.exists(MODEL_ID):
    print("계속 실행")
else:
    print("재확인 필요")

계속 실행


In [1]:
import os
import torch
import gc
import shutil
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from google.colab import files

# 1. 모델 설정
MODEL_ID = "/content/base_model"
OUT_DIR  = "./model"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"

NUM_CALIBRATION_SAMPLES = 512
MAX_SEQUENCE_LENGTH = 512

SCHEME = "W4A16"
TARGETS = ["Linear"]

# model.layers.X.mlp 하위의 모든 레이어를 보호함.
IGNORE_STRATEGY = [
    "embed_tokens",
    "lm_head",
    "re:model\.layers\.\d+\.mlp\..*"
]

# 2. 모델 로드
print("모델 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 3. 데이터셋 전처리
print("데이터 전처리 중...")
ds = load_dataset(DATASET_ID, split="train")
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    return {"text": tokenizer.apply_chat_template(
        example["conversations"],
        add_generation_prompt=True,
        tokenize=False
    )}

ds = ds.map(preprocess, remove_columns=ds.column_names)

gc.collect()
torch.cuda.empty_cache()

# 4. GPTQ 양자화
print(f"GPTQ 시작")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE_STRATEGY,
        dampening_frac=0.01,
        block_size=128
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

# 5. 저장 및 압축
print("저장 중")
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

zip_name = "submit1"
shutil.make_archive(base_name=zip_name, format="zip", root_dir=".", base_dir=OUT_DIR)
files.download(f"{zip_name}.zip")

print(f"[완료")

<>:26: SyntaxWarning: invalid escape sequence '\.'
<>:26: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-15575/1585629210.py:26: SyntaxWarning: invalid escape sequence '\.'
  "re:model\.layers\.\d+\.mlp\..*"


모델 로드 중...


`torch_dtype` is deprecated! Use `dtype` instead!


데이터 전처리 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Compressing model: 120it [00:02, 44.07it/s]
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[완료
